# 02 - Silver Transformation

## Objetivo

Realizar a limpeza, padronização e enriquecimento dos dados da camada Bronze.

## Atividades

- Ajuste de tipos de dados
- Padronização de colunas
- Enriquecimento com agrupamento de canais
- Tratamento de qualidade
- Persistência na camada Silver

In [0]:
# Importação das bibliotecas necessárias

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Leitura das tabelas da camada Bronze

sales_df = spark.table(
    "workspace.bronze.beverage_sales"
)

channel_df = spark.table(
    "workspace.bronze.beverage_channel_group"
)

print("Sales:", sales_df.count())
print("Channel:", channel_df.count())

Sales: 16151
Channel: 31


In [0]:
print(sales_df.columns)
print(channel_df.columns)

['DATE', 'CE_BRAND_FLVR', 'BRAND_NM', 'Btlr_Org_LVL_C_Desc', 'CHNL_GROUP', 'TRADE_CHNL_DESC', 'PKG_CAT', 'Pkg_Cat_Desc', 'TSR_PCKG_NM', 'sales_volume', 'YEAR', 'MONTH', 'PERIOD']
['TRADE_CHNL_DESC', 'TRADE_GROUP_DESC', 'TRADE_TYPE_DESC']


In [0]:
# Conversão dos tipos de dados para análise

sales_typed_df = (
    sales_df
    .withColumn(
        "DATE",
        to_date(col("DATE"), "M/d/yyyy")
    )
    .withColumn(
        "sales_volume",
        regexp_replace(col("sales_volume"), ",", ".").cast("double")
    )
    .withColumn(
        "YEAR",
        col("YEAR").cast("int")
    )
    .withColumn(
        "MONTH",
        col("MONTH").cast("int")
    )
    .withColumn(
        "PERIOD",
        col("PERIOD").cast("int")
    )
)

sales_typed_df.printSchema()

root
 |-- DATE: date (nullable = true)
 |-- CE_BRAND_FLVR: integer (nullable = true)
 |-- BRAND_NM: string (nullable = true)
 |-- Btlr_Org_LVL_C_Desc: string (nullable = true)
 |-- CHNL_GROUP: string (nullable = true)
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- PKG_CAT: string (nullable = true)
 |-- Pkg_Cat_Desc: string (nullable = true)
 |-- TSR_PCKG_NM: string (nullable = true)
 |-- sales_volume: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)



In [0]:
# Correção definitiva da data usando limpeza mais forte
# Remove qualquer caractere que não seja número ou barra

sales_typed_df = (
    sales_df
    .withColumn(
        "DATE",
        regexp_replace(col("DATE"), "[^0-9/]", "")
    )
    .withColumn(
        "DATE",
        to_date(col("DATE"), "M/d/yyyy")
    )
    .withColumn(
        "sales_volume",
        regexp_replace(col("sales_volume"), "[^0-9.-]", "").cast("double")
    )
    .withColumn(
        "YEAR",
        regexp_replace(col("YEAR").cast("string"), "[^0-9]", "").cast("int")
    )
    .withColumn(
        "MONTH",
        regexp_replace(col("MONTH").cast("string"), "[^0-9]", "").cast("int")
    )
    .withColumn(
        "PERIOD",
        regexp_replace(col("PERIOD").cast("string"), "[^0-9]", "").cast("int")
    )
)

display(sales_typed_df.select("DATE", "sales_volume", "YEAR", "MONTH", "PERIOD").limit(10))

DATE,sales_volume,YEAR,MONTH,PERIOD
2006-01-01,22.48,2006,1,1
2006-01-01,100.0,2006,1,1
2006-01-01,66.14,2006,1,1
2006-01-01,222.5,2006,1,1
2006-01-01,302.5,2006,1,1
2006-01-01,10.0,2006,1,1
2006-01-01,-4.22,2006,1,1
2006-01-01,59.95,2006,1,1
2006-01-01,300.0,2006,1,1
2006-01-01,85.0,2006,1,1


In [0]:
# Validação de qualidade dos dados

print("Total de registros:", sales_typed_df.count())

print("Datas nulas:")
print(
    sales_typed_df.filter(
        col("DATE").isNull()
    ).count()
)

print("Volume nulo:")
print(
    sales_typed_df.filter(
        col("sales_volume").isNull()
    ).count()
)

Total de registros: 16151
Datas nulas:
0
Volume nulo:
0


In [0]:
# Padronização das chaves de junção para evitar falhas por espaços ou caracteres invisíveis

sales_typed_df = sales_typed_df.withColumn(
    "TRADE_CHNL_DESC_JOIN",
    upper(trim(regexp_replace(col("TRADE_CHNL_DESC"), "\\s+", " ")))
)

channel_clean_df = channel_df.withColumn(
    "TRADE_CHNL_DESC_JOIN",
    upper(trim(regexp_replace(col("TRADE_CHNL_DESC"), "\\s+", " ")))
)

In [0]:
# Enriquecimento da base de vendas com a classificação de canais

sales_enriched_df = (
    sales_typed_df.alias("s")
    .join(
        channel_clean_df.alias("c"),
        on="TRADE_CHNL_DESC_JOIN",
        how="left"
    )
    .select(
        "s.*",
        "c.TRADE_GROUP_DESC",
        "c.TRADE_TYPE_DESC"
    )
)

display(sales_enriched_df.limit(10))

TRADE_CHNL_DESC_JOIN,DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,sales_volume,YEAR,MONTH,PERIOD,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,2006-01-01,3440,LEMON,CANADA,LEISURE,SPORT VENUE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,2006-01-01,3440,LEMON,NORTHEAST,SUPERS,SUPERETTE,N20O,20Z/600ML,20Z NRP 24L,100.0,2006,1,1,SERVICES,MIX
PLANT / OFFICE,2006-01-01,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,PLANT / OFFICE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1,SERVICES,MIX
MASS MERCHANDISER,2006-01-01,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1,GROCERY,MIX
MASS MERCHANDISER,2006-01-01,3440,LEMON,WEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,2006-01-01,3440,LEMON,MIDWEST,OTHER SMALL STORES,LIQUOR/BEER/WINE/SOFT,N20O,20Z/600ML,20Z NRP 24L,10.0,2006,1,1,GROCERY,ALCOHOLIC
CONVENIENCE STORE,2006-01-01,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,SERVICES,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,CANADA,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,2006-01-01,3554,STRAWBERRY,SOUTHWEST,SUPERS,SUPERMARKET,N20O,20Z/600ML,20z NRP 24L S,300.0,2006,1,1,GROCERY,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,20Z NRP 24L,85.0,2006,1,1,ENTERTAINMENT,ALCOHOLIC


In [0]:
# Verifica se existem canais sem correspondência na base auxiliar

qtd_sem_classificacao = sales_enriched_df.filter(
    col("TRADE_GROUP_DESC").isNull()
).count()

print("Registros sem classificação de canal:", qtd_sem_classificacao)

Registros sem classificação de canal: 0


In [0]:
# Persistência da tabela enriquecida na camada Silver

sales_enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.sales_enriched")

print("Tabela workspace.silver.sales_enriched criada com sucesso.")

Tabela workspace.silver.sales_enriched criada com sucesso.


In [0]:
# Validação final da camada Silver

silver_df = spark.table("workspace.silver.sales_enriched")

print("Quantidade de registros na Silver:", silver_df.count())

display(silver_df.limit(10))

Quantidade de registros na Silver: 16151


TRADE_CHNL_DESC_JOIN,DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,sales_volume,YEAR,MONTH,PERIOD,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,2006-01-01,3440,LEMON,CANADA,LEISURE,SPORT VENUE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,2006-01-01,3440,LEMON,NORTHEAST,SUPERS,SUPERETTE,N20O,20Z/600ML,20Z NRP 24L,100.0,2006,1,1,SERVICES,MIX
PLANT / OFFICE,2006-01-01,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,PLANT / OFFICE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1,SERVICES,MIX
MASS MERCHANDISER,2006-01-01,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1,GROCERY,MIX
MASS MERCHANDISER,2006-01-01,3440,LEMON,WEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,2006-01-01,3440,LEMON,MIDWEST,OTHER SMALL STORES,LIQUOR/BEER/WINE/SOFT,N20O,20Z/600ML,20Z NRP 24L,10.0,2006,1,1,GROCERY,ALCOHOLIC
CONVENIENCE STORE,2006-01-01,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,SERVICES,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,CANADA,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,2006-01-01,3554,STRAWBERRY,SOUTHWEST,SUPERS,SUPERMARKET,N20O,20Z/600ML,20z NRP 24L S,300.0,2006,1,1,GROCERY,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,20Z NRP 24L,85.0,2006,1,1,ENTERTAINMENT,ALCOHOLIC
